In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=r"icdn(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import optuna
from icdn import ICDNModel, ICDNConfig, PanelSchema
from icdn.data.splits import TemporalSplitter

ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent

DATASETS = {
    "walmart": {
        "path": ROOT / "data" / "M5-walmart" / "panel" / "m5_icdn_panel.parquet",
        "out": ROOT / "data" / "M5-walmart" / "panel" / "icdn-opt",
        "schema": PanelSchema(category="category"),
        "own_elasticity_bounds": (-3.5, 0.0),
        "cross_elasticity_bounds": (-0.4, 0.8),
        "beta_prior": -2.0,
        "same_category_first": True,
        "min_coverage": 0.5,
    },
    "one_c": {
        "path": ROOT / "data" / "predict-future-sales-1c" / "panel" / "1c_icdn_panel.parquet",
        "out": ROOT / "data" / "predict-future-sales-1c" / "panel" / "icdn-opt",
        "schema": PanelSchema(category="category"),
        "own_elasticity_bounds": (-3.0, 0.0),
        "cross_elasticity_bounds": (-0.2, 0.5),
        "beta_prior": -1.5,
        "same_category_first": True,
        "min_coverage": 0.15,
    },
}

N_FOLDS = 3
MIN_TRAIN_FRAC = 0.5
N_TRIALS = 15
SEED = 42
HIDDEN_CHOICES = {"256_128_64": (256, 128, 64), "128_64_32": (128, 64, 32), "128_64": (128, 64)}


def load_panel(spec):
    panel = pd.read_parquet(spec["path"])
    return panel[(panel["price"] > 0) & (panel["units"] > 0)].copy()


def prepare_folds(spec):
    panel = load_panel(spec)
    return TemporalSplitter(period_col="week_id").expanding_splits(
        panel, n_folds=N_FOLDS, min_train_frac=MIN_TRAIN_FRAC
    )


def suggest_icdn(trial, n_products):
    hidden_key = trial.suggest_categorical("hidden", list(HIDDEN_CHOICES))
    return dict(
        hidden=HIDDEN_CHOICES[hidden_key],
        dropout=trial.suggest_float("dropout", 0.05, 0.40),
        lr=trial.suggest_float("lr", 5e-4, 3e-3, log=True),
        warmup_lr=trial.suggest_float("warmup_lr", 5e-4, 3e-3, log=True),
        lambda_smooth=trial.suggest_float("lambda_smooth", 0.01, 0.08, log=True),
        lambda_elast=trial.suggest_float("lambda_elast", 0.01, 0.08, log=True),
        k_neighbors=min(trial.suggest_int("k_neighbors", 2, 4), max(n_products - 1, 1)),
        n_knots=trial.suggest_int("n_knots", 3, 5),
    )


def make_config(spec, n_products, searched):
    return ICDNConfig(
        schema=spec["schema"],
        n_products=n_products,
        k_neighbors=searched["k_neighbors"],
        same_category_first=spec["same_category_first"],
        own_elasticity_bounds=spec["own_elasticity_bounds"],
        cross_elasticity_bounds=spec["cross_elasticity_bounds"],
        beta_prior=spec["beta_prior"],
        min_coverage=spec["min_coverage"],
        hidden=searched["hidden"],
        dropout=searched["dropout"],
        lr=searched["lr"],
        warmup_lr=searched["warmup_lr"],
        lambda_smooth=searched["lambda_smooth"],
        lambda_elast=searched["lambda_elast"],
        n_knots=searched["n_knots"],
        enforce_negative_beta=True,
        warmup_epochs=15,
        epochs=40,
        early_stopping_patience=8,
        seed=SEED,
        verbose=False,
    )


def mae_one(train_raw, val_raw, spec, searched):
    n_products = int(train_raw["product_code"].nunique())
    model = ICDNModel(make_config(spec, n_products, searched))
    model.fit(train_raw)
    return float(model.evaluate(val_raw)["mae"])


def objective(trial, spec, folds):
    n_products = int(folds[0][0]["product_code"].nunique())
    searched = suggest_icdn(trial, n_products)
    maes = []
    for k, (train_raw, val_raw) in enumerate(folds):
        mae = mae_one(train_raw, val_raw, spec, searched)
        maes.append(mae)
        trial.set_user_attr(f"fold{k}_mae", mae)
        trial.report(float(np.mean(maes)), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(maes))


def dump_best(study, out_dir):
    best = dict(study.best_params)
    best["hidden"] = list(HIDDEN_CHOICES[best["hidden"]])
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "best_params.json").write_text(json.dumps(best, indent=2))
    study.trials_dataframe().to_csv(out_dir / "optuna_trials.csv", index=False)
    print("best MAE", study.best_value)
    print("best params", best)
    print("wrote", out_dir / "best_params.json")
    return best


def run_study(name, spec):
    print(f"\n=== {name} Optuna ===")
    folds = prepare_folds(spec)
    out_dir = spec["out"]
    out_dir.mkdir(parents=True, exist_ok=True)
    study = optuna.create_study(
        study_name=f"icdn_{name}",
        storage=f"sqlite:///{out_dir / 'optuna.db'}",
        load_if_exists=True,
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=0),
    )
    study.optimize(lambda t: objective(t, spec, folds), n_trials=N_TRIALS, gc_after_trial=True)
    return dump_best(study, out_dir)

/home/thebigmonster/Github/nn-elasticity-additional-work/venv-icdn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
best_by_dataset = {}
for name, spec in DATASETS.items():
    best_by_dataset[name] = run_study(name, spec)


=== walmart Optuna ===


[I 2026-08-25 12:08:12,794] A new study created in RDB with name: icdn_walmart
[I 2026-08-25 12:08:23,936] Trial 0 finished with value: 0.8179686268170675 and parameters: {'hidden': '128_64_32', 'dropout': 0.25953046946896285, 'lr': 0.0006612658646612707, 'warmup_lr': 0.0006612372870684138, 'lambda_smooth': 0.011283783077201558, 'lambda_elast': 0.06056685237625633, 'k_neighbors': 3, 'n_knots': 5}. Best is trial 0 with value: 0.8179686268170675.
[I 2026-08-25 12:08:37,650] Trial 1 finished with value: 0.707675019900004 and parameters: {'hidden': '128_64_32', 'dropout': 0.12431868873739667, 'lr': 0.0006925598810528834, 'warmup_lr': 0.0006945227129221505, 'lambda_smooth': 0.018826002986047252, 'lambda_elast': 0.02977846306223348, 'k_neighbors': 3, 'n_knots': 3}. Best is trial 1 with value: 0.707675019900004.
[I 2026-08-25 12:08:50,934] Trial 2 finished with value: 0.6496227582295736 and parameters: {'hidden': '256_128_64', 'dropout': 0.17822664515279213, 'lr': 0.0011320391141177721, 'warm

KeyboardInterrupt: 